
# Example: Apple production in Germany
This didactic example shows how to build a Life Cycle Inventory using the ecoinvent database.
In this simple example, we aim to calculate the environmental impact of 1 kg of apples produced in Germany.
The yield is 40 tonnes of apples per hectare.
The following information was collected:
Fertilizer consumption: 80 kg N/ha
Diesel consumption for machinery: 435 L/ha
Pesticide application: 3.36 kg/ha


In [ ]:
# Install the required dependencies

In [ ]:
!pip install bw2calc>=2.1 -q # Brightway package
!pip install bw2data>=4.5 -q # Brightway package
!pip install bw2io>=0.9.11 -q # Brightway package
!pip install bw2analyzer # Brightway package
!pip install pandas -q
!pip install pypardiso -q
!pip install mermaid-py -q # Este paquete permite construir diagramas. Lo usaré para el reporte.
!pip install seaborn>=0.13.2 -q

<div class="alert alert-block alert-warning">
⚠️ You must restart the session!
</div>

We need to download a backup file that contains ecoinvent. To do this, we need to authenticate the **Gmail** user that was created previously.

In [ ]:
from google.colab import auth
from oauth2client.client import GoogleCredentials
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
drive.CreateFile({"id": "1E3wPUOuRp13ucuNmq3557EuT3mszlmHB"}).GetContentFile(
    "backup.tar.gz"
)

Check the file.

In [ ]:
!du -hs backup.tar.gz

In [ ]:
import bw2calc as bc
import bw2data as bd
import bw2io as bi
from rich import print

### Import the project backup
This approach does not require much explanation: the project is loaded again.

In [ ]:
bi.restore_project_directory(
    "backup.tar.gz",  # nombre del archivo, creado celdas arriba
    project_name="proyecto_ei",  # Se puede elegir un nombre nuevo para el proyecto
    overwrite_existing=True,
)

In [ ]:
# Select the corresponding project and assign the databases to variables
bd.projects.set_current("proyecto_ei")
ei = bd.Database("ecoinvent-3.9.1-cutoff")
# Set the current project
bio = bd.Database("biosphere3")

In [ ]:
# Create a new database for our example
if "manzana_db" in bd.databases:
    del bd.databases["manzana_db"]
manzana_db = bd.Database("manzana_db")
manzana_db.register()

In [ ]:
list(manzana_db)

In [ ]:
# Create an activity called apple_germany
# Step 2: Create the main apple-production activity
manzana_alemania = manzana_db.new_activity(
    code="manzana_alemania",
    name="Producción de manzanas, frescas, en Alemania",
    location="DE",
    unit="kilogram",
    type="processwithreferenceproduct",
)
# Save the main activity
manzana_alemania.save()

### Search for the activities we want to connect

In [ ]:
# In this case, I want to search for an activity representing ammonium nitrate consumption as fertilizer
print(
    "------Activity------, -------Producto de referencia ------------,-----CODE--------"
)
for found in ei.search("ammonium nitrate"):
    print(f"{found} , {found['reference product']}, {found['code']}")

In [ ]:
# Now we choose the second element of the list (index 1) because it refers to RER (Rest of Europe).
# We choose Rest of Europe because it is the best approximation for `Germany`.
# We can inspect the content to verify that this is the activity we want to include as an input.
fertilizer = ei.get("78a08d1a028eef23c65c9475010c14bd")
# As you will see, 1kg de esta urea contiene alrededor de 0.301 kg de Nitrógeno N
# Taken from the comments...
# " ... If 1 kg of this 'urea ammonium nitrate mix' is used as fertiliser, it is equivalent to circa 0.301 kg of Nitrogen as N ..."
print(fertilizer.as_dict())

In [ ]:
manzana_alemania.new_exchange(
    amount=0.002,  # 80 kg N/ha con rendimiento de 40 t/ha de manzanas.
    type="technosphere",
    input=fertilizer,
).save()

In [ ]:
# We repeat the same search exercise for the pesticide.
print(
    "------Activity------, -------Producto de referencia ------------,-----CODE--------"
)
for found in ei.search("pesticide"):
    print(f"{found} , {found['reference product']}, {found['code']}")

In [ ]:
# We choose the pesticide produced in RER.
pesticide = ei.get("1c5c182327d3b34ec6a3db5024e60d1a")
print(pesticide.as_dict())

In [ ]:
manzana_alemania.new_exchange(
    amount=0.000084,  # 3.36 kg/ha con 40 t/ha de rendimiento
    type="technosphere",
    input=pesticide,
).save()

In [ ]:
# I took the liberty of choosing an activity that produces 1 kg of diesel.
diesel = ei.get("dd036517891922c427be648943a735a3")

# Diesel
manzana_alemania.new_exchange(
    amount=0.0090806,  # 435 L/ha con 40 t/ha rendimiento, densidad del diesel 0.835 kg/L
    type="technosphere",
    input=diesel,
).save()

In [ ]:
# Similarly, we know that diesel combustion generates CO2 emissions
# In this case, we search in the biosphere.
print("------Activity------, -----CODE--------")
for found in bio.search("carbon dioxide"):
    print(f"{found} , {found['code']}")

In [ ]:
# We choose the first element in the list because the CO2 emissions are of fossil origin.
co2 = bio.get("349b29d1-3e58-4c66-98b9-9d1a076efd2e")
print(co2.as_dict())

# Fossil CO2 emissions
manzana_alemania.new_exchange(
    amount=0.024,  # Asumimos que todo el diesel se convierte en CO2, por estequiometría 1 kg de diesel se convierte en 2.68 kg de CO2
    type="biosphere",
    input=co2,
).save()

In [ ]:
# We repeat this exercise y consideramos las emisiones de nitrógeno al suelo producto de la aplicación de fertilizantes

n_soil = bio.get("b748f6f1-7061-4243-89c7-3f2d01dcec07")
manzana_alemania.new_exchange(
    amount=0.0004214,  # Asumimos de manera simplificada que el 70% del nitrógeno se pierde y termina en el suelo.
    # (0.002 * 0.301 * 0.7) -> amount of fertilizer * nitrogen content per kg * loss percentage.
    type="biosphere",
    input=n_soil,
).save()

In [ ]:
# Now we inspect the flows of our activity
print(list(list(manzana_db)[0].exchanges()))

# Impact assessment methods
I want to analyze two impact methods: climate change and acidification. First, however, we need to know the exact method names.

In [ ]:
# A method is a tuple with the following structure: (<method>, <impact category>, <indicator>)
# Podemos explorar en la lista de methods disponibles de esta manera:
for method in bd.methods:
    if (
        "EF v3.1" in method[0]
    ):  # Queremos identificar todos los methods del Environmental Footprint Methodology.
        print(method)

In [ ]:
metodo_1 = ("IPCC 2021", "climate change", "global warming potential (GWP100)")
# Calculate the impact
lca_cc = bc.LCA({manzana_alemania: 1}, method=metodo_1)  # Instancia la clase
lca_cc.lci()  # calcula el inventario de ciclo de vida
lca_cc.lcia()  # Calcula los impactos
print("The climate change impact (kg CO2eq) es:")
print(lca_cc.score)

In [ ]:
# Now for acidification
metodo_2 = ("EF v3.1", "acidification", "accumulated exceedance (AE)")
lca_acid = bc.LCA({manzana_alemania: 1}, method=metodo_2)  # Instancia la clase
lca_acid.lci()  # calcula el inventario de ciclo de vida
lca_acid.lcia()  # Calcula los impactos
print("The acidification impact (mol H+ eq) es:")
print(lca_acid.score)

In [ ]:
# We perform a contribution analysis solo para el primer method como demostración.
import bw2analyzer as ba
import pandas as pd

print(
    pd.DataFrame(
        [
            (x, y, z["name"])
            for x, y, z in ba.ContributionAnalysis().annotated_top_processes(lca=lca_cc)
        ],
        columns=["score", "quantity", "name"],
    )
)

In [ ]:
import bw2analyzer as ba
import pandas as pd

# Create the dataframe with emissions data
print(
    pd.DataFrame(
        [
            (x, y, z["name"])
            for x, y, z in ba.ContributionAnalysis().annotated_top_emissions(lca=lca_cc)
        ],
        columns=["score", "quantity", "name"],
    )
)

Here you can see an example of how you can report your code:


# Life Cycle Assessment Report: Apple Production in Germany

## 1. Introduction

### 1.1 Functional unit
This inventory represents the production of 1 kg of fresh apples produced in Germany.
The assumed yield is 40 tonnes per hectare.

### 1.2 System boundaries
In this simplified example, only the following processes were considered:
- Fertilizer consumption (ammonium nitrate): 80 kg N/ha
- Diesel consumption for machinery: 435 L/ha
- Pesticide application: 3.36 kg/ha
- Direct CO2 emissions from diesel combustion
- Nitrogen emissions to soil from fertilizer losses

In [ ]:
# If you wish, you can import an image en vez de diagramar directamente en el notebook.
# I used a diagram-generation language called Mermaid:
# https://mermaid.live/edit#pako:eNpVkcFygjAQhl8ls2e0IUBQpuOMFb1pe-ipwCElK2aExAlh2ur4VH2EvliROm3NKbvffv8e9gSlkQgJkG1t3sqdsI48p7kmlzfPVmidqtVRaIf3r_ZutlHOCmeIRDJvjFamIKPRjKTZkzWyK0v19al7OMz6ZF-RtdC9LdrimvmQPWHrVKmkuKpXsMhShS3Wt910qJbZslGtMhpbsnhkxQ1c_YMbImrSdlibAjyorJKQbEXdogcN2kZcajhd9BzcDhvMIem_Uth9Drk-99JB6BdjGkic7XrNmq7a_YZ0BykcpkpUVvyNoJZoF6bTDhI_mAwZkJzgHRIWTscBD-PYD-iU8igOPPjo2z4dRyH3I8YZDUM64WcPjsPaHkwnMWURj33GOWXMA5TKGbv-udRwsPM3zCWGMA
import mermaid as md

render = md.Mermaid("""
    flowchart TD
         A[Fertilizante<br/>Nitrato de Amonio] --> D[Produccion de<br/>1 kg Manzanas]
         B[Pesticida] --> D
         C[Diesel] --> D
         D --> E[Emisiones CO2]
         D --> F[Emisiones N al suelo]

         classDef green fill:#90EE90
         class E,F green
""")
render


The following processes were excluded from the system: agricultural infrastructure, transport, processing, packaging, irrigation, labor, among others.

## 2. Life Cycle Inventory

### 2.1 Technosphere
**Fertilizers**: The ammonium nitrate production process from the RER region (Rest of Europe) was used. Consumption was calculated considering that 1 kg of ammonium nitrate mixture is equivalent to 0.301 kg of nitrogen.

**Pesticides**: The generic pesticide production process from the RER region was used. Consumption was approximated by dividing the application per hectare by the yield.

**Diesel**: The diesel production process was selected using a density of 0.835 kg/L to convert liters to kilograms.

### 2.2 Biosphere
**Carbon dioxide**: This flow was calculated using stoichiometry, assuming that all diesel is converted into CO2 (factor of 2.68 kg CO2/kg diesel).

**Nitrogen to soil**: It was estimated that 70% of the nitrogen applied as fertilizer is lost and ends up in the soil.

## 3. Impact assessment

### 3.1 Selected methods
**Climate change**: The IPCC 2021 climate change method with a 100-year global warming potential (GWP100) was selected. This method considers the most up-to-date characterization factors for greenhouse gases.

**Acidification**: The EF v3.1 (Environmental Footprint) method for acidification was used, with the accumulated exceedance (AE) indicator.

## 4. Results

For climate change, the total impact is approximately 0.035 kg CO2-eq per kg of apple. The contribution analysis indicates that the process with the largest direct impact is apple production itself, followed by emissions associated with natural gas and diesel production.

The substance emitted the most is fossil carbon dioxide (2.64e-02 kg CO2-eq), followed by fossil methane (3.47e-03 kg CO2-eq).

The processes with the highest contribution are:
1. Apple production (total impact): 0.024 kg CO2-eq
2. Natural gas venting: 0.003315 kg CO2-eq
3. Diesel production: 0.001774 kg CO2-eq
